# Chapitre 7 : Analyse inférentielle et test des hypothèses

**Auteur :** Ibrahima Baldé, Master 2 ASAD, USSEIN (Kaolack, Sénégal)
**Lien avec le mémoire :** Chapitre 7 (sections 7.1 à 7.3)
**Entrée :** `data/Base_Enquete_Kaolack_178.xlsx` (non publiée, voir README)
**Contenu :** régressions de comptage (Poisson / binomiale négative) de l'IAA sur HDDS et WDDS, médiation (modèle 4 de Hayes, bootstrap 5 000 réplications), modération.
**Résultats :** tableaux Excel et figures écrits dans `outputs/`.
**Exécution :** lancer les cellules dans l'ordre, de haut en bas.

In [ ]:
import os
os.makedirs("outputs", exist_ok=True)  # dossier des tableaux et figures


# CHAPITRE 7 : ANALYSE INFÉRENTIELLE ET TEST DES HYPOTHÈSES DE RECHERCHE

### Évaluation de l'impact des pratiques agroécologiques sur la sécurité nutritionnelle à Kaolack

**Données :** Base de 178 ménages agricoles (`Base_Enquete_Kaolack_178.xlsx`)  
---

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

# Chargement direct avec le chemin d'accès exact du Bureau
df = pd.read_excel("data/Base_Enquete_Kaolack_178.xlsx")

# Nettoyage des noms de colonnes
df.columns = df.columns.str.strip().str.lower()

print("✅ Succès ! Nombre de ménages :", len(df))

In [ ]:
# Affiche l'index (0, 1, 2...) et le nom exact de chaque colonne
for i, col in enumerate(df.columns):
    print(f"Index {i} : {col}")

In [ ]:
import re
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import poisson
import matplotlib.pyplot as plt

# Configuration des graphiques
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.family'] = 'sans-serif'

# =========================================================================
# 1. CHARGEMENT ET PRÉPARATION DES DONNÉES
# =========================================================================
file_path = "data/Base_Enquete_Kaolack_178.xlsx"
df = pd.read_excel(file_path)

def extraire_score_pratique(val):
    if pd.isna(val):
        return 0
    match = re.search(r'[0-3]', str(val).strip())
    return int(match.group(0)) if match else 0

def binariser_groupe_alimentaire(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip().lower()
    return 1 if val_str in ['1', '1.0', 'oui', 'yes', 'true'] else 0

# Recalcul de l'IAA (25 pratiques)
index_25_pratiques = [
    49, 70, 87, 106, 121, 134, 149, 166, 182, 197, 
    213, 230, 245, 260, 276, 292, 307, 323, 338, 354, 
    369, 384, 400, 416, 429
]
df_iaa_items = df.iloc[:, index_25_pratiques].map(extraire_score_pratique)
df['iaa_recalcule_brut'] = df_iaa_items.sum(axis=1)

# Recalcul HDDS et WDDS
groupes_hdds = df.iloc[:, 448:460].map(binariser_groupe_alimentaire)
df['hdds_recalcule'] = groupes_hdds.fillna(0).sum(axis=1)

groupes_wdds = df.iloc[:, 467:477].map(binariser_groupe_alimentaire)
df['wdds_recalcule'] = groupes_wdds.fillna(0).sum(axis=1)
if 'wdds_score_brut' in df.columns:
    df.loc[df['wdds_score_brut'].isna(), 'wdds_recalcule'] = np.nan

# Assemblage des 7 variables d'analyse
df_7vars = pd.DataFrame({
    'hdds': df['hdds_recalcule'],
    'wdds': df['wdds_recalcule'],
    'iaa': df['iaa_recalcule_brut'],
    'age_cm': pd.to_numeric(df.iloc[:, 14], errors='coerce'),
    'niveau_inst': df.iloc[:, 16].fillna("Aucun / Non scolarisé").astype(str),
    'taille_menage': pd.to_numeric(df.iloc[:, 18], errors='coerce'),
    'superficie': pd.to_numeric(df.iloc[:, 481], errors='coerce')
})

# Harmonisation des catégories du niveau d'instruction
df_7vars['niveau_inst'] = df_7vars['niveau_inst'].replace({
    '1': 'Primaire', '2': 'Secondaire', '3': 'Supérieur', '0': 'Aucun / Non scolarisé',
    'T.1': 'Primaire', 'T.2': 'Secondaire', 'T.3': 'Supérieur', 'Aucun': 'Aucun / Non scolarisé'
})

# =========================================================================
# 2. DIAGNOSTIC DE SURDISPERSION ET SÉLECTION DU MODÈLE (SECTION 7.1.1)
# =========================================================================
def diagnostic_comptage(data, var_dep):
    cols = [var_dep, 'iaa', 'age_cm', 'niveau_inst', 'taille_menage', 'superficie']
    df_mod = data[cols].dropna().copy()
    
    y = df_mod[var_dep]
    mean_y = y.mean()
    var_y = y.var()
    ratio_dispersion_brut = var_y / mean_y
    
    formule = f"{var_dep} ~ iaa + age_cm + C(niveau_inst) + taille_menage + superficie"
    
    # 1. Modèle Poisson
    mod_poisson = smf.glm(formula=formule, data=df_mod, family=sm.families.Poisson()).fit()
    pearson_chi2_p = mod_poisson.pearson_chi2 / mod_poisson.df_resid
    
    # 2. Modèle Binomial Négatif (NB2)
    mod_nb = smf.glm(formula=formule, data=df_mod, family=sm.families.NegativeBinomial()).fit()
    pearson_chi2_nb = mod_nb.pearson_chi2 / mod_nb.df_resid
    
    # Test de surdispersion de Cameron & Trivedi (1990)
    mu = mod_poisson.fittedvalues
    aux_y = ((y - mu)**2 - y) / mu
    aux_mod = sm.OLS(aux_y, mu).fit()
    p_val_surdispersion = aux_mod.pvalues[0]
    
    res_diag = {
        'Variable Dépendante': var_dep.upper(),
        'Echantillon (N)': len(df_mod),
        'Moyenne E(Y)': round(mean_y, 4),
        'Variance Var(Y)': round(var_y, 4),
        'Ratio Dispersion Brut (Var/E)': round(ratio_dispersion_brut, 4),
        'Poisson Pearson Chi2/df': round(pearson_chi2_p, 4),
        'Poisson Log-Likelihood': round(mod_poisson.llf, 2),
        'Poisson AIC': round(mod_poisson.aic, 2),
        'NB2 Pearson Chi2/df': round(pearson_chi2_nb, 4),
        'NB2 Log-Likelihood': round(mod_nb.llf, 2),
        'NB2 AIC': round(mod_nb.aic, 2),
        'p-value Test Surdispersion (alpha=0)': round(p_val_surdispersion, 4),
        'Modèle Optimal Sélectionné': 'Poisson GLM' if p_val_surdispersion > 0.05 and pearson_chi2_p < 1.2 else 'Binomiale Négative'
    }
    
    return res_diag, mod_poisson, mod_nb, df_mod

diag_hdds, mod_p_hdds, mod_nb_hdds, df_mod_hdds = diagnostic_comptage(df_7vars, 'hdds')
diag_wdds, mod_p_wdds, mod_nb_wdds, df_mod_wdds = diagnostic_comptage(df_7vars, 'wdds')

df_diagnostics = pd.DataFrame([diag_hdds, diag_wdds])

print("=" * 85)
print("📊 SECTION 7.1.1 : TABLEAU DE DIAGNOSTIC DE SURDISPERSION ET SÉLECTION DU MODÈLE")
print("=" * 85)
print(df_diagnostics.T.to_string())
print("=" * 85 + "\n")

# =========================================================================
# 3. ESTIMATION DES MODÈLES POISSON GLM HC3 FINAUX
# =========================================================================
def estimer_poisson_hc3_formate(data, var_dep):
    cols = [var_dep, 'iaa', 'age_cm', 'niveau_inst', 'taille_menage', 'superficie']
    df_mod = data[cols].dropna().copy()
    
    formule = f"{var_dep} ~ iaa + age_cm + C(niveau_inst) + taille_menage + superficie"
    mod = smf.glm(formula=formule, data=df_mod, family=sm.families.Poisson()).fit(cov_type='HC3')
    
    ci = mod.conf_int()
    res = pd.DataFrame({
        'Variable': mod.params.index,
        'Coef (Beta)': mod.params.values,
        'SE (HC3)': mod.bse.values,
        'z-value': mod.tvalues.values,
        'p-value': mod.pvalues.values,
        'IRR (exp(Beta))': np.exp(mod.params.values),
        'IC 95% Bas': np.exp(ci[0].values),
        'IC 95% Haut': np.exp(ci[1].values)
    })
    
    res['Variable'] = res['Variable'].str.replace('C(niveau_inst)[T.', '', regex=False).str.replace(']', '', regex=False)
    res['Significativité'] = res['p-value'].apply(lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns')))
    
    return mod, res

mod_hdds_hc3, res_hdds_df = estimer_poisson_hc3_formate(df_7vars, 'hdds')
mod_wdds_hc3, res_wdds_df = estimer_poisson_hc3_formate(df_7vars, 'wdds')

# =========================================================================
# 4. EXPORTATION DES TABLEAUX EXCEL MULTI-FEUILLES
# =========================================================================
output_excel = "outputs/Resultats_Section_7_1.xlsx"

with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
    df_diagnostics.to_excel(writer, sheet_name='7.1.1_Diagnostics', index=False)
    res_hdds_df.to_excel(writer, sheet_name='7.1_Poisson_HDDS_HC3', index=False)
    res_wdds_df.to_excel(writer, sheet_name='7.1_Poisson_WDDS_HC3', index=False)

print(f"✅ Tableaux d'analyse exportés avec succès : {output_excel}\n")

# =========================================================================
# 5. GÉNÉRATION DES GRAPHIQUES (AVEC SCIPI.STATS POISSON)
# =========================================================================

# FIGURE 1 : Diagnostic de Surdispersion (Observé vs Ajusté Poisson)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=300)

for ax, df_mod, mod_p, var_label in zip(
    axes, [df_mod_hdds, df_mod_wdds], [mod_p_hdds, mod_p_wdds], ['HDDS (Ménage)', 'WDDS (Femme)']
):
    obs = df_mod[var_label.split()[0].lower()].value_counts(normalize=True).sort_index()
    
    # Correction : utilisation directe de scipy.stats.poisson.pmf
    pred_p = pd.Series(mod_p.fittedvalues).apply(lambda m: poisson.pmf(obs.index, m)).mean(axis=0)
    
    ax.bar(obs.index - 0.15, obs.values, width=0.3, alpha=0.7, color='#1f77b4', label='Fréquences Observées')
    ax.plot(obs.index, pred_p, marker='o', linewidth=2, color='#d62728', label='Ajustement Poisson')
    
    ax.set_title(f'Diagnostic de distribution : Score {var_label}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Score de Diversité Alimentaire', fontsize=10)
    ax.set_ylabel('Proportion', fontsize=10)
    ax.legend(frameon=True)
    ax.set_xticks(obs.index)

plt.tight_layout()
fig1_path = "outputs/Diagnostic_Surdispersion_Distribution.png"
plt.savefig(fig1_path)
plt.close()

# FIGURE 2 : Effets Marginaux Prédits de l'IAA sur HDDS et WDDS
fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
iaa_range = np.linspace(df_7vars['iaa'].min(), df_7vars['iaa'].max(), 100)

def calculer_predictions_marginales(mod, data_mod, iaa_seq):
    preds = []
    for val_iaa in iaa_seq:
        df_temp = data_mod.copy()
        df_temp['iaa'] = val_iaa
        preds.append(mod.predict(df_temp).mean())
    return np.array(preds)

pred_hdds_m = calculer_predictions_marginales(mod_hdds_hc3, df_mod_hdds, iaa_range)
pred_wdds_m = calculer_predictions_marginales(mod_wdds_hc3, df_mod_wdds, iaa_range)

ax.plot(iaa_range, pred_hdds_m, label='HDDS Prédit (Ménage, N=178)', color='#1f77b4', linewidth=2.5)
ax.plot(iaa_range, pred_wdds_m, label='WDDS Prédit (Femme, N=145)', color='#2ca02c', linewidth=2.5, linestyle='--')

ax.set_title('Effet direct de l\'Agroécologie (IAA) sur la Diversité Alimentaire', fontsize=12, fontweight='bold')
ax.set_xlabel('Indice d\'Adoption Agroécologique (IAA - Score brut sur 75)', fontsize=11)
ax.set_ylabel('Score de Diversité Alimentaire Prédit (Nombre de groupes)', fontsize=11)
ax.legend(frameon=True, fontsize=10)

plt.tight_layout()
fig2_path = "outputs/Effet_Direct_IAA_Marginal.png"
plt.savefig(fig2_path)
plt.close()

# FIGURE 3 : Forest Plot des Ratios d'Incidence (IRR)
fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=300)

for ax, df_res, title in zip(axes, [res_hdds_df, res_wdds_df], ['HDDS (Ménage)', 'WDDS (Femme)']):
    df_plot = df_res[df_res['Variable'] != 'Intercept'].copy()
    y_pos = np.arange(len(df_plot))
    
    ax.errorbar(
        df_plot['IRR (exp(Beta))'], y_pos,
        xerr=[df_plot['IRR (exp(Beta))'] - df_plot['IC 95% Bas'], df_plot['IC 95% Haut'] - df_plot['IRR (exp(Beta))']],
        fmt='o', color='#1f77b4', ecolor='#d62728', elinewidth=2, capsize=4, ms=6
    )
    
    ax.axvline(x=1.0, color='black', linestyle=':', alpha=0.7)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(df_plot['Variable'], fontsize=10)
    ax.set_title(f'Ratios d\'Incidence (IRR) - Modèle {title}', fontsize=11, fontweight='bold')
    ax.set_xlabel('IRR (exp(Beta)) et IC à 95%', fontsize=10)
    ax.invert_yaxis()

plt.tight_layout()
fig3_path = "outputs/ForestPlot_IRR_HDDS_WDDS.png"
plt.savefig(fig3_path)
plt.close()

print("✅ Les 3 graphiques ont été générés sans erreur et enregistrés dans votre dossier !")

In [ ]:
import re
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# Fixer la graine pour reproductibilité du bootstrap
np.random.seed(12345)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# -------------------------------------------------------------------------
# 1. CHARGEMENT ET EXTRACTION STRICTE DES 5 VARIABLES
# -------------------------------------------------------------------------
file_path = "data/Base_Enquete_Kaolack_178.xlsx"
df = pd.read_excel(file_path)

def extraire_score_pratique(val):
    if pd.isna(val): return 0
    match = re.search(r'[0-3]', str(val).strip())
    return int(match.group(0)) if match else 0

def binariser_groupe_alimentaire(val):
    if pd.isna(val): return np.nan
    val_str = str(val).strip().lower()
    return 1 if val_str in ['1', '1.0', 'oui', 'yes', 'true'] else 0

# Variables X (iaa) et Y (hdds)
index_25_pratiques = [
    49, 70, 87, 106, 121, 134, 149, 166, 182, 197, 
    213, 230, 245, 260, 276, 292, 307, 323, 338, 354, 
    369, 384, 400, 416, 429
]
df['iaa'] = df.iloc[:, index_25_pratiques].map(extraire_score_pratique).sum(axis=1)
df['hdds'] = df.iloc[:, 448:460].map(binariser_groupe_alimentaire).fillna(0).sum(axis=1)

# Variables M1, M2 et Strate
milieu_urbain = df.iloc[:, 12].astype(str).str.strip()
df['milieu'] = np.where((milieu_urbain.notna()) & (~milieu_urbain.isin(['nan', 'None', '', '0'])), 1, 0)
df['autoconsommation'] = pd.to_numeric(df.iloc[:, 464], errors='coerce').fillna(0)  # M1
df['revenu_agricole'] = pd.to_numeric(df.iloc[:, 491], errors='coerce').fillna(0) / 100000  # M2

df_clean = df[['hdds', 'iaa', 'autoconsommation', 'revenu_agricole', 'milieu']].dropna().copy()

# -------------------------------------------------------------------------
# 2. MODÈLE 4 DE HAYES (ESTIMATION ET BOOTSTRAP 5 000 RÉPLIQUES)
# -------------------------------------------------------------------------
def executer_modele4(data, nom_echantillon, n_boot=5000):
    n = len(data)
    
    # Équations OLS avec erreurs robustes HC3
    mod_m1 = smf.ols('autoconsommation ~ iaa', data=data).fit(cov_type='HC3')
    mod_m2 = smf.ols('revenu_agricole ~ iaa', data=data).fit(cov_type='HC3')
    mod_y  = smf.ols('hdds ~ iaa + autoconsommation + revenu_agricole', data=data).fit(cov_type='HC3')
    
    ind_m1_pt = mod_m1.params['iaa'] * mod_y.params['autoconsommation']
    ind_m2_pt = mod_m2.params['iaa'] * mod_y.params['revenu_agricole']
    
    boot_m1, boot_m2 = [], []
    for _ in range(n_boot):
        sample = data.sample(n=n, replace=True)
        m1_b = smf.ols('autoconsommation ~ iaa', data=sample).fit()
        m2_b = smf.ols('revenu_agricole ~ iaa', data=sample).fit()
        y_b  = smf.ols('hdds ~ iaa + autoconsommation + revenu_agricole', data=sample).fit()
        
        boot_m1.append(m1_b.params['iaa'] * y_b.params['autoconsommation'])
        boot_m2.append(m2_b.params['iaa'] * y_b.params['revenu_agricole'])
        
    def stats_boot(effets):
        moy = np.mean(effets)
        ic_bas = np.percentile(effets, 2.5)
        ic_haut = np.percentile(effets, 97.5)
        p_val = 2 * min((np.array(effets) > 0).mean(), (np.array(effets) < 0).mean())
        sig = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else 'ns'))
        return moy, ic_bas, ic_haut, p_val, sig

    m1_moy, m1_bas, m1_haut, m1_p, m1_sig = stats_boot(boot_m1)
    m2_moy, m2_bas, m2_haut, m2_p, m2_sig = stats_boot(boot_m2)
    
    resultats = [
        {
            'Strate': nom_echantillon, 'Médiateur': 'M1 (Autoconsommation)',
            'Effet Point': round(ind_m1_pt, 4), 'Effet Boot': round(m1_moy, 4),
            'IC 95% Bas': round(m1_bas, 4), 'IC 95% Haut': round(m1_haut, 4),
            'p-value': round(m1_p, 4), 'Sig': m1_sig
        },
        {
            'Strate': nom_echantillon, 'Médiateur': 'M2 (Revenu Agricole)',
            'Effet Point': round(ind_m2_pt, 4), 'Effet Boot': round(m2_moy, 4),
            'IC 95% Bas': round(m2_bas, 4), 'IC 95% Haut': round(m2_haut, 4),
            'p-value': round(m2_p, 4), 'Sig': m2_sig
        }
    ]
    return resultats, mod_m1, mod_m2, mod_y

# Exécutions
res_g, m1_g, m2_g, y_g = executer_modele4(df_clean, "Global (N=178)")
res_p, _, _, _          = executer_modele4(df_clean[df_clean['milieu'] == 0], "Périurbain (n=89)")
res_u, _, _, _          = executer_modele4(df_clean[df_clean['milieu'] == 1], "Urbain (n=89)")

df_effets = pd.DataFrame(res_g + res_p + res_u)

# -------------------------------------------------------------------------
# 3. EXPORT DES DEUX FICHIERS EXCEL
# -------------------------------------------------------------------------
# Fichier 1 : Synthèse des Effets Indirects Bootstrap
excel_effets_path = "outputs/Resultats_Modele4_Effets_Indirects.xlsx"
with pd.ExcelWriter(excel_effets_path, engine='openpyxl') as writer:
    df_effets.to_excel(writer, sheet_name='Effets_Indirects_M4', index=False)

# Fichier 2 : Détail des Équations OLS (Global)
excel_eq_path = "outputs/Resultats_Modele4_Equations_OLS.xlsx"
with pd.ExcelWriter(excel_eq_path, engine='openpyxl') as writer:
    pd.DataFrame(m1_g.params, columns=['Coef']).to_excel(writer, sheet_name='Eq_M1_Autoconsommation')
    pd.DataFrame(m2_g.params, columns=['Coef']).to_excel(writer, sheet_name='Eq_M2_Revenu')
    pd.DataFrame(y_g.params, columns=['Coef']).to_excel(writer, sheet_name='Eq_Y_HDDS')

print(f"✅ Fichier Excel 1 créé : {excel_effets_path}")
print(f"✅ Fichier Excel 2 créé : {excel_eq_path}")

# -------------------------------------------------------------------------
# 4. GÉNÉRATION ET SAUVEGARDE DU GRAPHIQUE
# -------------------------------------------------------------------------
categories = ['Zone Périurbaine (n=89)', 'Zone Urbaine (n=89)']
x = np.arange(len(categories))
width = 0.35

df_m1_sub = df_effets[(df_effets['Médiateur'] == 'M1 (Autoconsommation)') & (df_effets['Strate'] != 'Global (N=178)')].reset_index(drop=True)
df_m2_sub = df_effets[(df_effets['Médiateur'] == 'M2 (Revenu Agricole)') & (df_effets['Strate'] != 'Global (N=178)')].reset_index(drop=True)

m1_eff = df_m1_sub['Effet Boot'].tolist()
m1_err = [
    [e - b for e, b in zip(m1_eff, df_m1_sub['IC 95% Bas'])],
    [h - e for e, h in zip(m1_eff, df_m1_sub['IC 95% Haut'])]
]

m2_eff = df_m2_sub['Effet Boot'].tolist()
m2_err = [
    [e - b for e, b in zip(m2_eff, df_m2_sub['IC 95% Bas'])],
    [h - e for e, h in zip(m2_eff, df_m2_sub['IC 95% Haut'])]
]

fig, ax = plt.subplots(figsize=(9, 5.5), dpi=300)

ax.bar(x - width/2, m1_eff, width, label='Canal Autoconsommation (M1)', color='#3182bd', yerr=m1_err, capsize=5, ecolor='#1c3d5a')
ax.bar(x + width/2, m2_eff, width, label='Canal Revenu Agricole (M2)', color='#74c476', yerr=m2_err, capsize=5, ecolor='#2ca25f')

for i in range(len(categories)):
    ax.text(x[i] - width/2, m1_eff[i] + max(m1_err[1][i], 0) + 0.008, df_m1_sub.loc[i, 'Sig'], ha='center', va='bottom', fontweight='bold', color='#3182bd', fontsize=11)
    ax.text(x[i] + width/2, m2_eff[i] + max(m2_err[1][i], 0) + 0.008, df_m2_sub.loc[i, 'Sig'], ha='center', va='bottom', fontweight='bold', color='#2ca25f', fontsize=11)

ax.set_ylabel("Magnitude de l'Effet Indirect (Beta)", fontsize=11, fontweight='bold')
ax.set_title("Comparaison des Effets Indirects selon la Strate (Modèle 4 Brut)", fontsize=12, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=10, fontweight='bold')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.legend(frameon=True, facecolor='white', edgecolor='none')
ax.grid(axis='y', linestyle=':', alpha=0.6)

plt.tight_layout()
fig_path = "outputs/Graphique_Modele4_Brut.png"
plt.savefig(fig_path, dpi=300)
plt.show()

print(f"✅ Graphique PNG créé : {fig_path}")

In [ ]:
import pandas as pd

# 1. Charger les deux fichiers
df = pd.read_excel("data/Base_Enquete_Kaolack_178.xlsx")
df_qgis = pd.read_csv(
    "data/distance_marche_kaolack.csv",
    names=['fid', 'id_menage', 'lat', 'lon', 'HubName', 'HubDist'],
    header=0
)

# 2. Convertir la distance en kilomètres (HubDist / 1000)
df_qgis['distance_marche'] = pd.to_numeric(df_qgis['HubDist'], errors='coerce') / 1000

# 3. Nettoyer les espaces invisibles autour des identifiants
df['Nouveau_id'] = df['Nouveau_id'].astype(str).str.strip()
df_qgis['id_menage'] = df_qgis['id_menage'].astype(str).str.strip()

# 4. Fusionner les deux bases : Nouveau_id (Excel) <---> id_menage (QGIS)
df_final = pd.merge(
    df,
    df_qgis[['id_menage', 'distance_marche']],
    left_on='Nouveau_id',
    right_on='id_menage',
    how='left'
)

# 5. Vérifier les 10 premières lignes et enregistrer le résultat
print(df_final[['Nouveau_id', 'distance_marche']].head(10))
df_final.to_excel("outputs/donnees_completes_analyse.xlsx", index=False)
print(df_final['distance_marche'].describe())
df_final['distance_marche_centree'] = df_final['distance_marche'] - df_final['distance_marche'].mean()

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# ==============================================================================
# 1. EXTRACTION DES VARIABLES D'INTÉRÊT
# ==============================================================================

# Variable Indépendante (X) : IAA (Index de 25 pratiques)
index_25_pratiques = [
    49, 70, 87, 106, 121, 134, 149, 166, 182, 197, 
    213, 230, 245, 260, 276, 292, 307, 323, 338, 354, 
    369, 384, 400, 416, 429
]
df_final['iaa'] = df_final.iloc[:, index_25_pratiques].map(extraire_score_pratique).sum(axis=1)

# Variable Dépendante (Y) : HDDS (12 groupes alimentaires)
df_final['hdds'] = df_final.iloc[:, 448:460].map(binariser_groupe_alimentaire).fillna(0).sum(axis=1)

# Modérateur Biophysique (W1) : Salinité des sols (Index 496)
df_final['salinite_sols'] = pd.to_numeric(df_final.iloc[:, 496], errors='coerce').fillna(0)

# Modérateur Géographique (W2) : Distance au marché (issue du réseau QGIS)
df_final['distance_marche'] = pd.to_numeric(df_final['distance_marche'], errors='coerce')


# ==============================================================================
# 2. CALCUL DES VARIABLES CENTRÉES ET TERMES D'INTERACTION
# ==============================================================================

# Centrage sur les moyennes
df_final['IAA_centre'] = df_final['iaa'] - df_final['iaa'].mean()
df_final['Salinite_centree'] = df_final['salinite_sols'] - df_final['salinite_sols'].mean()
df_final['Distance_centree'] = df_final['distance_marche'] - df_final['distance_marche'].mean()

# Création des interactions
df_final['IAA_x_Salinite'] = df_final['IAA_centre'] * df_final['Salinite_centree']
df_final['IAA_x_Distance'] = df_final['IAA_centre'] * df_final['Distance_centree']


# ==============================================================================
# 3. ESTIMATION DES MODÈLES DE MODÉRATION (H3 - MODÈLE 1 DE HAYES)
# ==============================================================================

# Modèle 1a : Modération Biophysique (Salinité)
modele_salinite = smf.ols(
    'hdds ~ IAA_centre + Salinite_centree + IAA_x_Salinite', 
    data=df_final
).fit()

# Modèle 1b : Modération Géographique (Distance)
modele_distance = smf.ols(
    'hdds ~ IAA_centre + Distance_centree + IAA_x_Distance', 
    data=df_final
).fit()

# Modèle 1c : Modèle Territorial Complet (Section 7.3.1)
modele_territorial_complet = smf.ols(
    'hdds ~ IAA_centre + Salinite_centree + Distance_centree + IAA_x_Salinite + IAA_x_Distance', 
    data=df_final
).fit()


# ==============================================================================
# 4. AFFICHAGE DES RÉSULTATS
# ==============================================================================

print("=== MODÈLE 1a : SALINITÉ DES SOLS ===")
print(modele_salinite.summary())

print("\n=== MODÈLE 1b : DISTANCE AU MARCHÉ ===")
print(modele_distance.summary())

print("\n=== MODÈLE GLOBAL 7.3.1 : CONTRAINTES TERRITORIALES ===")
print(modele_territorial_complet.summary())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration du style graphique pour le mémoire
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 11, 'font.family': 'sans-serif'})

# ==============================================================================
# 1. EXPORTATION DES DONNÉES COMPLÈTES DANS EXCEL
# ==============================================================================

chemin_excel = "outputs/Base_Section_7_3_1_Hayes.xlsx"
df_final.to_excel(chemin_excel, index=False)
print(f" Base Excel exportée avec succès : {chemin_excel}")


# ==============================================================================
# 2. GRAPHIQUE 1 : MODÉRATION PAR LA SALINITÉ DES SOLS (Modèle 1a)
# ==============================================================================

fig, ax = plt.subplots(figsize=(8, 5))

# Calcul de l'écart-type
sd_sal = df_final['Salinite_centree'].std()
x_vals = np.linspace(df_final['IAA_centre'].min(), df_final['IAA_centre'].max(), 100)

# Coefficients du modèle 1a
b0 = modele_salinite.params['Intercept']
b_x = modele_salinite.params['IAA_centre']
b_w = modele_salinite.params['Salinite_centree']
b_xw = modele_salinite.params['IAA_x_Salinite']

# Génération des droites aux 3 seuils de salinité (-1 SD, Moyenne, +1 SD)
niveaux_sal = [
    ('Salinité faible (-1 SD)', -sd_sal, '#2ca02c'),
    ('Salinité moyenne (0 SD)', 0, '#ff7f0e'),
    ('Salinité élevée (+1 SD)', sd_sal, '#d62728')
]

for label, w_val, color in niveaux_sal:
    y_vals = b0 + (b_x * x_vals) + (b_w * w_val) + (b_xw * x_vals * w_val)
    ax.plot(x_vals, y_vals, label=label, color=color, linewidth=2.5)

ax.set_title("Effet modérateur de la salinité des sols ($H_3$ - Modèle 1a)", fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel("Indice d'Adoption Agroécologique centré (IAA)", fontsize=11)
ax.set_ylabel("Score de Diversité Alimentaire (HDDS)", fontsize=11)
ax.legend(title="Contrainte Salinité", frameon=True)
plt.tight_layout()

chemin_graph1 = "outputs/graphique_moderation_salinite.png"
plt.savefig(chemin_graph1, dpi=300)
plt.show()
print(f" Graphique Salinité enregistré : {chemin_graph1}")


# ==============================================================================
# 3. GRAPHIQUE 2 : MODÉRATION PAR LA DISTANCE AU MARCHÉ (Modèle 1b)
# ==============================================================================

fig, ax = plt.subplots(figsize=(8, 5))

sd_dist = df_final['Distance_centree'].std()

# Coefficients du modèle 1b
b0_d = modele_distance.params['Intercept']
b_x_d = modele_distance.params['IAA_centre']
b_w_d = modele_distance.params['Distance_centree']
b_xw_d = modele_distance.params['IAA_x_Distance']

# Génération des droites aux 3 seuils de distance (-1 SD, Moyenne, +1 SD)
niveaux_dist = [
    ('Proche du marché (-1 SD)', -sd_dist, '#1f77b4'),
    ('Distance moyenne (0 SD)', 0, '#ff7f0e'),
    ('Éloigné du marché (+1 SD)', sd_dist, '#9467bd')
]

for label, w_val, color in niveaux_dist:
    y_vals = b0_d + (b_x_d * x_vals) + (b_w_d * w_val) + (b_xw_d * x_vals * w_val)
    ax.plot(x_vals, y_vals, label=label, color=color, linewidth=2.5)

ax.set_title("Effet modérateur de la distance au marché ($H_3$ - Modèle 1b)", fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel("Indice d'Adoption Agroécologique centré (IAA)", fontsize=11)
ax.set_ylabel("Score de Diversité Alimentaire (HDDS)", fontsize=11)
ax.legend(title="Isolement Géographique", frameon=True)
plt.tight_layout()

chemin_graph2 = "outputs/graphique_moderation_distance.png"
plt.savefig(chemin_graph2, dpi=300)
plt.show()
print(f" Graphique Distance enregistré : {chemin_graph2}")

In [ ]:
import pandas as pd

# Chemin du fichier Excel de sortie
chemin_excel_tableaux = "outputs/Tableaux_Regressions_Chapitre_7.xlsx"

def extraire_metriques_natives(model):
    """Extrait directement les coefficients et statistiques depuis le modèle statsmodels."""
    # 1. Table des coefficients
    conf_int = model.conf_int()
    df_coef = pd.DataFrame({
        'coef': model.params,
        'std err': model.bse,
        't': model.tvalues,
        'P>|t|': model.pvalues,
        '[0.025': conf_int[0],
        '0.975]': conf_int[1]
    })
    
    # 2. Table des statistiques globales
    df_stats = pd.DataFrame({
        'Statistique': ['R-squared', 'Adj. R-squared', 'F-statistic', 'Prob (F-statistic)', 'AIC', 'BIC', 'No. Observations'],
        'Valeur': [
            round(model.rsquared, 4),
            round(model.rsquared_adj, 4),
            round(model.fvalue, 3),
            model.f_pvalue,
            round(model.aic, 2),
            round(model.bic, 2),
            int(model.nobs)
        ]
    })
    return df_coef, df_stats

# Création du fichier Excel multi-onglets
with pd.ExcelWriter(chemin_excel_tableaux, engine='openpyxl') as writer:
    
    # --------------------------------------------------------------------------
    # 1. TABLEAU COMPARATIF RÉCAPITULATIF
    # --------------------------------------------------------------------------
    variables_toutes = [
        'Intercept', 'IAA_centre', 'Salinite_centree', 
        'Distance_centree', 'IAA_x_Salinite', 'IAA_x_Distance'
    ]
    
    df_recap = pd.DataFrame(index=variables_toutes)
    
    modeles = [
        ('Modèle 1a (Salinité)', modele_salinite),
        ('Modèle 1b (Distance)', modele_distance),
        ('Modèle Global 7.3.1', modele_territorial_complet)
    ]
    
    for nom, mod in modeles:
        df_recap[f"{nom} - Coef"] = mod.params.round(4)
        df_recap[f"{nom} - Std Err"] = mod.bse.round(4)
        df_recap[f"{nom} - p-value"] = mod.pvalues.round(4)
        
    # Métriques de qualité d'ajustement en bas de tableau
    metriques = pd.DataFrame({
        'Modèle 1a (Salinité) - Coef': [round(modele_salinite.rsquared, 3), round(modele_salinite.rsquared_adj, 3), int(modele_salinite.nobs)],
        'Modèle 1b (Distance) - Coef': [round(modele_distance.rsquared, 3), round(modele_distance.rsquared_adj, 3), int(modele_distance.nobs)],
        'Modèle Global 7.3.1 - Coef': [round(modele_territorial_complet.rsquared, 3), round(modele_territorial_complet.rsquared_adj, 3), int(modele_territorial_complet.nobs)]
    }, index=['R²', 'R² Ajusté', 'Observations (N)'])
    
    df_recap_complet = pd.concat([df_recap, metriques])
    df_recap_complet.to_excel(writer, sheet_name="Synthèse Comparée")

    # --------------------------------------------------------------------------
    # 2. ONGLETS DÉTAILLÉS INDIVIDUELS
    # --------------------------------------------------------------------------
    for nom_feuille, mod in [
        ("Modèle 1a - Salinité", modele_salinite),
        ("Modèle 1b - Distance", modele_distance),
        ("Modèle 7.3.1 - Global", modele_territorial_complet)
    ]:
        df_coef, df_stats = extraire_metriques_natives(mod)
        
        # Écriture dans la feuille Excel
        df_coef.round(4).to_excel(writer, sheet_name=nom_feuille, startrow=0)
        df_stats.to_excel(writer, sheet_name=nom_feuille, startrow=len(df_coef) + 3, index=False)

print(f" Fichier Excel généré sans erreur : {chemin_excel_tableaux}")

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns

# ==============================================================================
# 1. CRÉATION DES VARIABLES ET DÉFINITION DU MÉDIATEUR (H4)
# ==============================================================================

# Nom de la variable médiatrice continue ('autoconsommation_centree' ou 'revenu_centree')
MEDIATEUR = 'autoconsommation_centree'

# Variable X : IAA
if 'iaa' not in df.columns:
    index_25_pratiques = [
        49, 70, 87, 106, 121, 134, 149, 166, 182, 197, 
        213, 230, 245, 260, 276, 292, 307, 323, 338, 354, 
        369, 384, 400, 416, 429
    ]
    df['iaa'] = df.iloc[:, index_25_pratiques].map(extraire_score_pratique).sum(axis=1)

# Variable M1 : Autoconsommation (colonne index 464)
if 'autoconsommation' not in df.columns:
    df['autoconsommation'] = pd.to_numeric(df.iloc[:, 464], errors='coerce').fillna(0)

# Variable M2 : Revenu agricole (colonne index 491)
if 'revenu_agricole' not in df.columns:
    df['revenu_agricole'] = pd.to_numeric(df.iloc[:, 491], errors='coerce').fillna(0) / 100000

# Variable Y : WDDS (colonnes index 467 à 477)
if 'wdds_recalcule' not in df.columns:
    groupes_wdds = df.iloc[:, 467:477].map(binariser_groupe_alimentaire)
    df['wdds_recalcule'] = groupes_wdds.fillna(0).sum(axis=1)

# Variable W : Pouvoir de décision féminin (colonne index 489)
def encoder_decision_femme(valeur):
    if pd.isna(valeur):
        return np.nan
    return 1 if valeur in [1, 3] else 0

if 'decision_femme' not in df.columns:
    df['decision_femme'] = df.iloc[:, 489].apply(encoder_decision_femme)

# Centrage des variables continues
df['iaa_centree'] = df['iaa'] - df['iaa'].mean()
df['autoconsommation_centree'] = df['autoconsommation'] - df['autoconsommation'].mean()
df['revenu_centree'] = df['revenu_agricole'] - df['revenu_agricole'].mean()

print(" Variables créées et variable 'MEDIATEUR' définie avec succès !")

# Nettoyage de l'échantillon d'analyse
cols_h4 = ['iaa_centree', MEDIATEUR, 'decision_femme', 'wdds_recalcule']
df_h4 = df.dropna(subset=cols_h4).copy()

# ==============================================================================
# 2. ESTIMATION DES ÉQUATIONS DU MODÈLE 14 DE HAYES
# ==============================================================================

# Étape 1 : Équation du Médiateur (M = i_M + a * X)
formule_m = f"{MEDIATEUR} ~ iaa_centree"
modele_m = smf.ols(formule_m, data=df_h4).fit()

# Étape 2 : Équation du WDDS (Y = i_Y + c'*X + b1*M + b2*W + b3*M*W)
formule_y = f"wdds_recalcule ~ iaa_centree + {MEDIATEUR} * decision_femme"
modele_y = smf.ols(formule_y, data=df_h4).fit()

# Extraction des coefficients
a = modele_m.params['iaa_centree']
c_prime = modele_y.params['iaa_centree']
b1 = modele_y.params[MEDIATEUR]
b2 = modele_y.params['decision_femme']
b3 = modele_y.params[f"{MEDIATEUR}:decision_femme"]

# Effets conditionnels
effet_indirect_w0 = a * (b1 + b3 * 0)
effet_indirect_w1 = a * (b1 + b3 * 1)
index_moderated_mediation = a * b3

# ==============================================================================
# 3. BOOTSTRAPPING (5 000 RÉÉCHANTILLONNAGES POUR IC 95%)
# ==============================================================================

n_boot = 5000
boot_w0, boot_w1, boot_index = [], [], []
np.random.seed(42)

for _ in range(n_boot):
    sample = df_h4.sample(frac=1.0, replace=True)
    
    mod_m_b = smf.ols(formule_m, data=sample).fit()
    a_b = mod_m_b.params['iaa_centree']
    
    mod_y_b = smf.ols(formule_y, data=sample).fit()
    b1_b = mod_y_b.params[MEDIATEUR]
    b3_b = mod_y_b.params[f"{MEDIATEUR}:decision_femme"]
    
    boot_w0.append(a_b * (b1_b + b3_b * 0))
    boot_w1.append(a_b * (b1_b + b3_b * 1))
    boot_index.append(a_b * b3_b)

ci_w0 = np.percentile(boot_w0, [2.5, 97.5])
ci_w1 = np.percentile(boot_w1, [2.5, 97.5])
ci_index = np.percentile(boot_index, [2.5, 97.5])

print("=" * 65)
print("RÉSULTATS DU MODÈLE 14 DE HAYES (MÉDIATION MODÉRÉE - H4)")
print("=" * 65)
print(f"Effet indirect (Décision faible W=0) : {effet_indirect_w0:.4f} | IC 95%: [{ci_w0[0]:.4f}, {ci_w0[1]:.4f}]")
print(f"Effet indirect (Décision forte W=1)  : {effet_indirect_w1:.4f} | IC 95%: [{ci_w1[0]:.4f}, {ci_w1[1]:.4f}]")
print(f"Indice de Médiation Modérée (a * b3) : {index_moderated_mediation:.4f} | IC 95%: [{ci_index[0]:.4f}, {ci_index[1]:.4f}]")
print("=" * 65)

# ==============================================================================
# 4. EXPORTATION DES TABLEAUX VERS EXCEL
# ==============================================================================

chemin_excel = "outputs/Tableaux_Regressions_Chapitre_7.xlsx"

with pd.ExcelWriter(chemin_excel, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    
    df_m_res = pd.DataFrame({
        'Coef': modele_m.params,
        'Std Err': modele_m.bse,
        't': modele_m.tvalues,
        'P>|t|': modele_m.pvalues.round(4)
    })
    df_m_res.round(4).to_excel(writer, sheet_name="H4 - Étape 1 (Médiateur)")

    df_y_res = pd.DataFrame({
        'Coef': modele_y.params,
        'Std Err': modele_y.bse,
        't': modele_y.tvalues,
        'P>|t|': modele_y.pvalues.round(4)
    })
    df_y_res.round(4).to_excel(writer, sheet_name="H4 - Étape 2 (WDDS)")

    df_boot_summary = pd.DataFrame({
        'Effet': ['Effet indirect (W=0 - Faible)', 'Effet indirect (W=1 - Fort)', 'Indice de Médiation Modérée'],
        'Estimation Ponctuelle': [effet_indirect_w0, effet_indirect_w1, index_moderated_mediation],
        'IC 95% Inférieur': [ci_w0[0], ci_w1[0], ci_index[0]],
        'IC 95% Supérieur': [ci_w0[1], ci_w1[1], ci_index[1]]
    })
    df_boot_summary.round(4).to_excel(writer, sheet_name="H4 - Effets Conditionnels", index=False)

print(f" Tables enregistrées dans le fichier Excel : {chemin_excel}")

# ==============================================================================
# 5. GÉNÉRATION DU GRAPHIQUE D'INTERACTION (Figure 7.3)
# ==============================================================================

sns.set_theme(style="whitegrid")
plt.figure(figsize=(9, 6), dpi=300)

m_range = np.linspace(df_h4[MEDIATEUR].min(), df_h4[MEDIATEUR].max(), 100)

pred_w0 = modele_y.params['Intercept'] + b1 * m_range
pred_w1 = modele_y.params['Intercept'] + b2 + (b1 + b3) * m_range

plt.plot(m_range, pred_w0, color='#d95f02', linewidth=2.5, label='Faible pouvoir de décision (W = 0)')
plt.plot(m_range, pred_w1, color='#1b9e77', linewidth=2.5, label='Fort pouvoir de décision (W = 1)')

plt.title("Effet d'interaction du pouvoir de décision féminin ($H_4$ - Modèle 14)", fontsize=12, fontweight='bold', pad=15)
plt.xlabel("Autoconsommation centrée (Médiateur M)", fontsize=11)
plt.ylabel("Score de Diversité Alimentaire des Femmes (WDDS)", fontsize=11)
plt.legend(title="Autonomie de Décision Féminine", frameon=True, fontsize=10)
plt.tight_layout()

chemin_graphique = "outputs/graphique_moderation_genre.png"
plt.savefig(chemin_graphique, dpi=300)
plt.show()
chemin_excel = "outputs/Tableaux_Regressions_Chapitre_7_v2.xlsx"

print(f" Figure 7.3 enregistrée : {chemin_graphique}")